In [2]:
import sys
from pathlib import Path
NOTEBOOKS = Path.cwd()                              # <-- Edit as needed
NEURALNET = NOTEBOOKS.parents[0]                    # <-- Edit as needed
BAMBOO_SETUP = NOTEBOOKS.parents[2]                 # <-- Edit as needed
Z_OUTPUT_eos = Path('/Users/antonett/Documents/HH Analysis/Z_OUTPUT_eos')   # <-- Edit as needed
sys.path.append(str((BAMBOO_SETUP/'src').resolve()))
import pandas as pd
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', 1000)  # Set a larger width to fit the editor window
# pd.set_option('display.max_colwidth', None)  # Allow columns to be fully displayed
%load_ext autoreload

In [4]:
from NeuralNet.DataHandler import DataHandler
import logging
datahandler = DataHandler(
    workdir=Z_OUTPUT_eos/'2022_even_1013/Reco',
    tree_name='SL_res_2b_x',
    total_inputs= NEURALNET / 'input/vars40_new.txt',
    log_level=logging.WARNING
)

total_df_unprep = datahandler.load_data()
total_df_unprep = datahandler.fix_column_names_mismatch(total_df_unprep)
datahandler.data_inspection(total_df_unprep)
df = datahandler.preprocess_data(total_df_unprep)
datahandler.data_inspection(df)
datahandler.data_summary(total_df_unprep)

		Number of duplicate events found: 8747
		Number of events with negative genWeights: 14290
		Number of events with potential outliers: 83738


# Set up your model config

In [4]:
from NeuralNet.utils import ModelConfig
model_config = ModelConfig(
    name='multi_HH_ttbar_tW',
    type='multi',
    categorization={"HH": ["HH_bbWW"], "ttbar": ["ttbar"], "tW": ["tW"]},
    training_weight_sf={"HH_bbWW": 1.0, "ttbar": 8.0, "tW": 4.0},
    input_vars='All',
    architecture_in_yml=True,
    residual_network=True,
    hiddenlayers=[
        {"type": 'Dense', "units": 64, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4},
        {"type": 'Dense', "units": 64, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4},
        {"type": 'Dense', "units": 64, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4}
    ],
    outputlayers=[
        {"type": 'Dense', "units": 3, "kernel_initializer": 'normal', "activation": 'softmax', "act_regularizer": {'l2': 1e-4}, "name": 'output'}
    ],
    compiler={"optimizer": 'adam', "lr": 0.001, "loss": 'categorical_crossentropy'},
    fit={"batch_size": 1024, "epochs": 35, "validation_split": 0.25}
)

In [32]:
%autoreload 2

# Run DNN

In [17]:
%autoreload 2

In [61]:
from NeuralNet.DNNModel import DNNModel, draw_all_stats
from tensorflow.keras.layers import Input, Masking, Normalization

class DNNModel_Custom():

    def Run_given_model(self, compiled_model):
        
        self.model = compiled_model

        self.train_model(X_train, Y_train, sw_train)
        self.save_model_info(X_train.columns, Y_train, Y_test)
        output_df, model_metrics = self.evaluate_and_predict(X_test, Y_test, evs_test)
        cm_norm_true, cm_norm_pred, diag_names = draw_all_stats(DNN_type=self.type, history=self.history, output_df=output_df, modeldir=self.modeldir, classes=self.classes)



DNN = DNNModel_Custom(model_config=model_config, modeldir= NOTEBOOKS/'model_custom', log_level=logging.DEBUG)
model_df = DNN.set_model_df_from_total_df(df)
X_train, X_test, Y_train, Y_test, evs_test, sw_train = DNN.Full_Splitting(model_df)


def model1(X_train):

    ndim = len(X_train.columns)
    input_layer = Input(shape=(ndim, ), name="input")
    normalizer = Normalization(
                    mean=X_train.mean(axis=0).to_numpy(),
                    variance=X_train.var(axis=0).to_numpy(),
                    name='normalization')
    normalized_input = normalizer(input_layer)




Initializing model: multi_HH_ttbar_tW
HH_bbWW
Total sum of genWeights for HH_bbWW: 137.84710693359375 
Total sum of sample_weights for HH_bbWW: 2340784.0 
ttbar
Total sum of genWeights for ttbar: 414462528.0 
Total sum of sample_weights for ttbar: 18726300.0 
tW
Total sum of genWeights for tW: 10632820.0 
Total sum of sample_weights for tW: 9363136.0 
